# Multi-Objective Optimization with MOGnowee

This notebook demonstrates how to use MOGnowee, a multi-objective extension of Gnowee that implements NSGA-II algorithm features. We'll solve a bi-objective spring optimization problem:

1. Minimize spring weight
2. Minimize spring deflection

The design variables are:
- D: Wire diameter
- W: Mean coil diameter
- L: Length of spring

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from MOGnowee import MOGnowee
from MOObjectiveFunction import MOObjectiveFunction
from ObjectiveFunction import ObjectiveFunction
from GnoweeUtilities import ProblemParameters

## Define Objective Functions

We'll define two objective functions for our spring optimization:

In [ ]:
def spring_weight(u):
    """Spring weight objective function."""
    assert len(u) == 3, ('Spring design needs to specify D, W, and L and '
                         'only those 3 parameters.')
    assert u[0] != 0 and u[1] != 0 and u[2] != 0, ('Design values {} '
                                             'cannot be zero.'.format(u))
    return ((2+u[2])*u[0]**2*u[1])

def spring_deflection(u):
    """Spring deflection objective function."""
    D, W, L = u
    # Simple spring deflection formula: F*L/(G*d^4)
    F = 100  # Applied force
    G = 11.5e6  # Shear modulus
    return (F*L)/(G*D**4)

# Create objective functions
obj1 = ObjectiveFunction(spring_weight)
obj2 = ObjectiveFunction(spring_deflection)

# Create multi-objective function
mo_obj = MOObjectiveFunction([obj1, obj2])

## Set Up Problem Parameters

Define the design space and constraints:

In [ ]:
# Problem parameters
lb = [0.05, 0.25, 2.0]  # Lower bounds [D, W, L]
ub = [2.0, 1.3, 15.0]   # Upper bounds [D, W, L]
varType = ['c']*3       # All continuous variables

# Create problem parameters object
prob = ProblemParameters(
    objective=mo_obj,
    lowerBounds=lb,
    upperBounds=ub,
    varType=varType
)

## Create and Run MOGnowee Optimizer

Initialize the optimizer with desired parameters and run the optimization:

In [ ]:
# Create MOGnowee instance
optimizer = MOGnowee(
    population=50,      # Population size
    maxGens=100,       # Maximum generations
    maxFevals=5000,    # Maximum function evaluations
    stallLimit=20,     # Stall limit
    optConvTol=1e-6,   # Convergence tolerance
    pps=prob           # Problem parameters
)

# Run optimization
timeline = optimizer.main()

## Analyze Results

Extract and visualize the Pareto front:

In [ ]:
# Get Pareto front solutions
pareto_front = []
for event in timeline:
    if isinstance(event.fitness, np.ndarray):  # Multi-objective solution
        pareto_front.append((event.design, event.fitness))

# Print results
print(f"\nFound {len(pareto_front)} Pareto optimal solutions:")
for i, (design, objectives) in enumerate(pareto_front):
    print(f"\nSolution {i+1}:")
    print(f"Design variables: D={design[0]:.6f}, W={design[1]:.6f}, L={design[2]:.6f}")
    print(f"Objectives: Weight={objectives[0]:.6f}, Deflection={objectives[1]:.6f}")

## Visualize Pareto Front

Plot the Pareto front to show the trade-off between objectives:

In [ ]:
# Extract objective values
weights = [obj[1][0] for obj in pareto_front]
deflections = [obj[1][1] for obj in pareto_front]

# Create scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(weights, deflections, c='blue', marker='o')
plt.xlabel('Spring Weight')
plt.ylabel('Spring Deflection')
plt.title('Pareto Front: Spring Weight vs Deflection')
plt.grid(True)

# Add annotations for some points
for i in range(min(5, len(pareto_front))):
    plt.annotate(f'Solution {i+1}', 
                 (weights[i], deflections[i]),
                 xytext=(10, 10), 
                 textcoords='offset points')

plt.show()

## Analyze Trade-offs

Let's analyze the trade-offs between objectives by looking at the extreme solutions:

In [ ]:
# Find solutions with minimum weight and minimum deflection
min_weight_idx = np.argmin(weights)
min_defl_idx = np.argmin(deflections)

print("Solution with minimum weight:")
design, obj = pareto_front[min_weight_idx]
print(f"Design variables: D={design[0]:.6f}, W={design[1]:.6f}, L={design[2]:.6f}")
print(f"Weight={obj[0]:.6f}, Deflection={obj[1]:.6f}")

print("\nSolution with minimum deflection:")
design, obj = pareto_front[min_defl_idx]
print(f"Design variables: D={design[0]:.6f}, W={design[1]:.6f}, L={design[2]:.6f}")
print(f"Weight={obj[0]:.6f}, Deflection={obj[1]:.6f}")

## Conclusion

The results show the trade-off between spring weight and deflection:
- Lighter springs tend to have more deflection
- Stiffer springs (less deflection) tend to be heavier
- The Pareto front represents the best possible trade-offs between these competing objectives

The designer can choose a solution from the Pareto front based on their specific requirements and preferences.